# 02 - Your first image classifier: a linear model

**Goal:** train a model yourself, understand every training step, and explain its predictions.
You will turn a 28 x 28 image into ten class scores, calculate a loss, inspect a gradient,
and keep the model that performs best on validation data.

**Before you start:** complete the environment and tensor/data lessons (00 and 01).
Know Python variables, loops, functions, and lists. No previous machine learning is needed.
This notebook also runs independently: choose **Kernel > Restart Kernel and Run All Cells**.

**Live demonstration:** allow 45-60 minutes including discussion. Training on the default
subset is much shorter; download speed, hardware, and package startup affect runtime.
`QUICK=True` uses a small real dataset. Smoke mode is only a plumbing check, not an experiment.

**Audience prediction:** can a weighted sum of pixels distinguish a sandal from a shirt?
Write down your prediction before training. Accuracy is measured here, never promised.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project root or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the complete course folder; course_utils.py is required.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import course_utils as cu

DEVICE = cu.configure(seed=42)
SMOKE = cu.SMOKE  # COURSE_SMOKE=1: validation only; too small to judge model quality.
QUICK = True     # Set False for a longer learning experiment, then Restart Kernel and Run All.


## 1. Choose an experiment size

An **epoch** is one pass through the training subset. A **batch** is a small group of images
processed together. Start small so you can see a complete experiment during a demonstration.
Change `QUICK` in the initialization cell above, then rerun the notebook, for a longer experiment.


In [ ]:
from copy import deepcopy
from torch import nn
from torchvision import transforms

SEED = 42
cu.seed_everything(SEED)
TRAIN_N = 96 if SMOKE else (3000 if QUICK else 48000)
VAL_N = 48 if SMOKE else (800 if QUICK else 12000)
TEST_N = 48 if SMOKE else (1000 if QUICK else 10000)
EPOCHS = 1 if SMOKE else (4 if QUICK else 15)
BATCH_SIZE = 64
LR = 0.05
print({"device": str(DEVICE), "train": TRAIN_N, "epochs": EPOCHS,
       "quick": QUICK, "smoke": SMOKE})


## 2. Load three separate datasets

FashionMNIST contains small grayscale clothing images. A label is an integer from 0 to 9.
Training images teach the model; validation images choose its checkpoint; test images
estimate performance only after that choice is finished.

`ToTensor` converts a PIL image into a float tensor with shape `[1, 28, 28]` and values in
`[0, 1]`. `Normalize((0.5,), (0.5,))` applies `(pixel - 0.5) / 0.5`, giving `[-1, 1]`.
These fixed constants are an explicit scaling choice, not statistics calculated from test data.
The helper makes a reproducible train/validation split from the original training set.


In [ ]:
preprocessing = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
train_ds, val_ds, test_ds, class_names = cu.fashion_data(
    train_transform=preprocessing, eval_transform=preprocessing,
    train_limit=TRAIN_N, val_limit=VAL_N, test_limit=TEST_N, seed=SEED,
)
train_loader = cu.loader(train_ds, batch_size=BATCH_SIZE, shuffle=True, seed=SEED)
val_loader = cu.loader(val_ds, batch_size=BATCH_SIZE)
test_loader = cu.loader(test_ds, batch_size=BATCH_SIZE)
print("Train / validation / test:", len(train_ds), len(val_ds), len(test_ds))
print(dict(enumerate(class_names)))


## 3. Inspect a batch before building a model

PyTorch image batches use **NCHW**: batch size, channels, height, width.
Labels use `torch.int64` because each number is a class index.
Always inspect real inputs before training: a mistaken shape or label can spoil the entire run.

**Ask the audience:** why is there one channel here, but usually three in a color photograph?


In [ ]:
images, labels = next(iter(train_loader))
print("Images:", images.shape, images.dtype)
print("Labels:", labels.shape, labels.dtype)
print("Range:", float(images.min()), float(images.max()))
assert images.shape[1:] == (1, 28, 28)
assert labels.dtype == torch.long
cu.show_images(train_ds, class_names, n=12, mean=(0.5,), std=(0.5,))


## 4. Convert pixels into ten scores

A linear classifier flattens `[N, 1, 28, 28]` into `[N, 784]`. For each class it learns
784 pixel weights and one bias. Its calculation is `scores = pixels @ weights.T + bias`.
The ten output numbers are **logits**: unrestricted scores, not probabilities.

The model has `(784 + 1) * 10 = 7,850` trainable numbers. This is multiclass logistic
regression when trained with cross-entropy, even though the PyTorch layer is called `Linear`.


In [ ]:
model = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 10)).to(DEVICE)
logits = model(images.to(DEVICE))
print(model)
print("Batch output:", logits.shape)
print("First image's logits:", logits[0].detach().cpu().numpy().round(2))
print("Parameters:", sum(p.numel() for p in model.parameters()))
assert logits.shape == (len(images), 10)


## 5. Understand the loss and the prediction

`CrossEntropyLoss` accepts raw logits and integer labels. It combines a numerically stable
log-softmax with negative log likelihood. **Do not add softmax before this loss.**
Lower loss means that the model assigns higher probability to the correct labels overall.

For display, softmax turns logits into values that sum to one. `argmax` chooses the largest
score, and gives the same class before or after softmax. A large softmax value alone does not
establish that the prediction is trustworthy or that the model is well calibrated.


In [ ]:
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, labels.to(DEVICE))
probabilities = logits.detach().softmax(dim=1).cpu()
predicted = logits.detach().argmax(dim=1).cpu()
print("Untrained batch loss:", round(loss.item(), 4))
print("Probability sum:", probabilities[0].sum().item())
print("Prediction:", class_names[predicted[0].item()])
print("Actual:", class_names[labels[0].item()])
assert torch.equal(predicted, probabilities.argmax(dim=1))


## 6. Perform exactly one learning step

Follow these five actions aloud: clear old gradients, run the model, calculate the loss,
calculate gradients with `backward`, and update weights with `step`.
A gradient tells us how a small weight change affects the current loss.

We use a **copy** for this demonstration so the main model still starts from its original
weights. A single update is not expected to solve the task or reduce every future batch loss.


In [ ]:
probe = deepcopy(model)
probe_optimizer = torch.optim.SGD(probe.parameters(), lr=LR)
x, y = images.to(DEVICE), labels.to(DEVICE)
before_weights = probe[1].weight.detach().clone()

probe_optimizer.zero_grad(set_to_none=True)
probe_logits = probe(x)
probe_loss = criterion(probe_logits, y)
probe_loss.backward()
print("Gradient norm:", probe[1].weight.grad.norm().item())
probe_optimizer.step()

print("Weight change norm:", (probe[1].weight - before_weights).norm().item())
with torch.no_grad():
    print("Same batch loss before / after:", probe_loss.item(), criterion(probe(x), y).item())
del probe, probe_optimizer


## 7. Establish a simple validation baseline

Before investing in a neural network, ask how well a trivial rule performs.
This baseline always predicts the most common **training** class. We assess it on validation
labels, without using test labels or changing the training set.
These teaching subsets are stratified, so class counts are equal or nearly equal.
This deliberately simple baseline gives you a reference point before learning starts.


In [ ]:
train_labels = torch.tensor([train_ds[i][1] for i in range(len(train_ds))])
majority_class = torch.bincount(train_labels, minlength=10).argmax().item()
val_labels = torch.tensor([val_ds[i][1] for i in range(len(val_ds))])
baseline_accuracy = (val_labels == majority_class).float().mean().item()
print("Always predict:", class_names[majority_class])
print(f"Validation baseline accuracy: {baseline_accuracy:.1%}")


## 8. Train with a fully visible loop

`model.train()` enables training behavior; `model.eval()` enables evaluation behavior.
This linear network has no dropout, but the distinction matters in later models.
`torch.no_grad()` stops gradient recording during validation. Validation never calls
`backward()` or `optimizer.step()`.

We multiply each batch's mean loss by its size before summing, then divide by the number
of examples. That handles a smaller final batch correctly. The best checkpoint is chosen
using **validation loss**, and copied so later updates cannot overwrite it.


In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=LR)
history = []
best_val_loss = float("inf")
best_state = deepcopy(model.state_dict())

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum, train_correct, train_count = 0.0, 0, 0
    for batch_images, batch_labels in train_loader:
        batch_images = batch_images.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        batch_logits = model(batch_images)
        batch_loss = criterion(batch_logits, batch_labels)
        batch_loss.backward()
        optimizer.step()
        count = batch_labels.size(0)
        train_loss_sum += batch_loss.item() * count
        train_correct += (batch_logits.argmax(1) == batch_labels).sum().item()
        train_count += count

    model.eval()
    val_loss_sum, val_correct, val_count = 0.0, 0, 0
    with torch.no_grad():
        for batch_images, batch_labels in val_loader:
            batch_images = batch_images.to(DEVICE)
            batch_labels = batch_labels.to(DEVICE)
            batch_logits = model(batch_images)
            batch_loss = criterion(batch_logits, batch_labels)
            count = batch_labels.size(0)
            val_loss_sum += batch_loss.item() * count
            val_correct += (batch_logits.argmax(1) == batch_labels).sum().item()
            val_count += count

    row = {"epoch": epoch, "train_loss": train_loss_sum / train_count,
           "val_loss": val_loss_sum / val_count,
           "train_accuracy": train_correct / train_count,
           "val_accuracy": val_correct / val_count}
    history.append(row)
    if row["val_loss"] < best_val_loss:
        best_val_loss = row["val_loss"]
        best_state = deepcopy(model.state_dict())
    print(f"Epoch {epoch:02d}: train loss={row['train_loss']:.3f}, "
          f"val loss={row['val_loss']:.3f}, val accuracy={row['val_accuracy']:.1%}")

model.load_state_dict(best_state)
print("Restored the checkpoint with lowest validation loss.")


## 9. Read the learning curves

A decreasing training loss shows that optimization is working on training data.
A growing gap between training and validation performance may indicate overfitting.
One short run cannot establish a reliable trend.

Training metrics above were collected while the weights changed across batches;
validation metrics use one fixed model at the end of the epoch. They are related,
but are not calculated under identical conditions.


In [ ]:
cu.plot_history(history)
selected_epoch = min(history, key=lambda row: row["val_loss"])["epoch"]
print("Selected epoch:", selected_epoch)
print(f"Validation majority baseline: {baseline_accuracy:.1%}")
print(f"Selected validation accuracy: {history[selected_epoch - 1]['val_accuracy']:.1%}")


## 10. See what the linear model learned

Each row of the weight matrix can be reshaped into a 28 x 28 image.
Red and blue indicate weights with different signs under the chosen color scale.
A weight contributes together with the pixel value, the bias, and competing class scores.
These pictures are model parameters, not a complete explanation of an individual prediction.


In [ ]:
weight_images = model[1].weight.detach().cpu().view(10, 28, 28)
scale = weight_images.abs().max().item()
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for class_id, ax in enumerate(axes.flat):
    ax.imshow(weight_images[class_id], cmap="coolwarm", vmin=-scale, vmax=scale)
    ax.set_title(class_names[class_id])
    ax.axis("off")
fig.suptitle("One learned pixel-weight map per class")
plt.tight_layout()
plt.show()


## 11. Evaluate the selected model on the held-out test set

This is the first model assessment using the test set. Once you have seen these results,
do not repeatedly change settings to improve this same test score. Develop changes using
validation data, and use a fresh holdout for a new final estimate when needed.
A subset score describes this sample, not every possible clothing photograph.


In [ ]:
test_logits, test_labels = cu.predict(model, test_loader, DEVICE)
test_predictions = test_logits.argmax(dim=1)
test_accuracy = (test_predictions == test_labels).float().mean().item()
test_loss = criterion(test_logits, test_labels).item()
print(f"Held-out test accuracy: {test_accuracy:.1%} on {len(test_labels)} images")
print(f"Held-out test loss: {test_loss:.4f}")

fig, axes = plt.subplots(2, 4, figsize=(11, 5))
for index, ax in enumerate(axes.flat):
    image, actual = test_ds[index]
    guess = test_predictions[index].item()
    ax.imshow((image[0] * 0.5 + 0.5).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"True: {class_names[actual]}\nPred: {class_names[guess]}",
                 color="green" if guess == actual else "crimson", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 12. Practice and teach it back

1. Calculate the parameter count for RGB 32 x 32 images and 5 classes without running code.
2. Repeat the training experiment with learning rates `0.005` and `0.05`, rebuilding the
   model and optimizer from the same seed each time. Compare validation loss only.
3. Remove `zero_grad()` from a **temporary copy** of the single-step demonstration. Run
   backward twice and inspect the gradient. Why do normal training loops clear it?
4. Explain why the model receives `[N, 784]` inputs but the loss receives `[N, 10]` logits.

<details><summary>Solution hints - open after trying</summary>

The RGB model has `(3 * 32 * 32 + 1) * 5 = 15,365` parameters.
Rebuild with `cu.seed_everything(SEED)` followed by a new `nn.Sequential(...)` and optimizer
before each learning-rate run; also recreate the shuffled loader with the same seed.
Gradients accumulate by default, so missing `zero_grad()` changes the intended update.
The 784 inputs are image features; the ten outputs are one score for each possible class.

```python
rgb_linear = nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 5))
print(sum(p.numel() for p in rgb_linear.parameters()))
```

</details>

**Presenter checkpoint:** have a learner narrate the five learning actions and point to
the exact line that prevents validation data from changing the weights.

**Limitations:** this model does not explicitly exploit neighboring pixels; these low-resolution
examples do not establish performance on real product photos. Next, add nonlinear features.

**References:** [PyTorch optimization tutorial](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html),
[CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html),
[FashionMNIST dataset](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.FashionMNIST.html).
